# AlphaGalerkin: Resolution-Independent Go AI

**Train once, play anywhere** — from 9×9 to 19×19 without retraining.

---

## What Makes AlphaGalerkin Different?

Traditional Go AIs (like AlphaGo) use **Convolutional Neural Networks (CNNs)** that are locked to a specific board size. Train on 19×19? You can only play 19×19.

AlphaGalerkin uses **Continuous Operator Learning** — treating the board as a continuous field rather than discrete pixels. This unlocks:

| Feature | Traditional CNNs | AlphaGalerkin |
|---------|------------------|---------------|
| Board Size | Fixed (e.g., 19×19) | Any size (5×5 → 25×25+) |
| Transfer | Retrain required | Zero-shot |
| Complexity | O(N²) attention | O(N) Galerkin attention |
| Speed | Standard | FFT-accelerated rollouts |

## Setup

First, let's import the key components and set up our environment.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

# AlphaGalerkin components
from config.schemas import OperatorConfig
from src.modeling.model import AlphaGalerkinModel, AlphaGalerkinFast
from src.modeling.attention import GalerkinAttention, SoftmaxAttention
from src.modeling.embeddings import FourierFeatures
from src.math_kernel.basis import FourierBasis, create_grid_coordinates
from src.physics.poisson import PoissonSolver

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---

# Part 1: The Magic of Resolution Independence

Let's see how the same model handles different board sizes without any modifications.

In [ ]:
# Create a compact model configuration
config = OperatorConfig(
    d_model=64,
    n_heads=4,
    n_galerkin_layers=2,
    n_softmax_layers=1,
    n_fourier_features=32,
    input_channels=17,  # Standard Go input channels
)

# Build the model
model = AlphaGalerkinModel(config).to(device)
model.eval()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Test on multiple board sizes with the SAME model weights
board_sizes = [5, 9, 13, 19]
results = {}

print("Testing resolution independence...\n")
print(f"{'Board Size':^12} | {'Input Shape':^18} | {'Policy Shape':^14} | {'Value':^10}")
print("-" * 60)

with torch.no_grad():
    for size in board_sizes:
        # Create random board state (batch=1, channels=17, height, width)
        x = torch.randn(1, 17, size, size).to(device)
        
        # Forward pass — same model, different sizes!
        output = model(x)
        
        # Extract outputs
        policy = output.policy_logits  # Move probabilities
        value = output.value           # Position evaluation
        
        results[size] = {
            'policy_size': policy.shape[-1],
            'value': value.item()
        }
        
        print(f"{size}×{size:^8} | {str(x.shape):^18} | {policy.shape[-1]:^14} | {value.item():^10.4f}")

print("\n✓ Same model works on all board sizes!")

### Why does this work?

AlphaGalerkin treats positions as **continuous coordinates** rather than discrete grid cells:

1. **Fourier Features** encode positions as waves, not pixels
2. **Galerkin Attention** approximates integral operators (like Green's functions)
3. **No fixed-size layers** — everything adapts to the input resolution

Let's visualize how Fourier Features work:

In [ ]:
# Visualize Fourier Features at different resolutions
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

fourier_encoder = FourierBasis(n_features=8, scale=1.0)

for idx, size in enumerate([5, 9, 13, 19]):
    # Create grid coordinates (normalized to [0, 1])
    coords = create_grid_coordinates(size, batch_size=1, device='cpu')
    
    # Get Fourier features
    features = fourier_encoder(coords)  # Shape: (1, size*size, n_features*2)
    
    # Reshape to grid
    feature_grid = features[0, :, 0].reshape(size, size).numpy()  # First feature
    
    # Plot the feature map
    ax = axes[0, idx]
    im = ax.imshow(feature_grid, cmap='RdBu', aspect='equal')
    ax.set_title(f'{size}×{size} Board', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Plot another feature for variety
    feature_grid2 = features[0, :, 5].reshape(size, size).numpy()
    ax2 = axes[1, idx]
    ax2.imshow(feature_grid2, cmap='viridis', aspect='equal')
    ax2.set_xticks([])
    ax2.set_yticks([])

axes[0, 0].set_ylabel('Feature 1\n(cos wave)', fontsize=10)
axes[1, 0].set_ylabel('Feature 6\n(sin wave)', fontsize=10)

fig.suptitle('Fourier Features at Different Resolutions\n(Same pattern, different samplings)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---

# Part 2: Galerkin Attention — O(N) Global Influence

Traditional attention is O(N²) — every token attends to every other token. 
**Galerkin Attention** achieves O(N) by using mathematical projections from numerical analysis.

The key insight: Instead of computing N² attention weights, we:
1. Project Values onto a compact Key basis
2. Multiply by Queries to reconstruct

This is mathematically equivalent to approximating a **Green's function** — how influence propagates across the board.

In [ ]:
# Compare Galerkin vs Softmax attention
d_model = 64
n_heads = 4

galerkin_attn = GalerkinAttention(d_model, n_heads)
softmax_attn = SoftmaxAttention(d_model, n_heads)

# Benchmark on increasing sequence lengths
seq_lengths = [25, 81, 169, 361, 625]  # 5², 9², 13², 19², 25² (board positions)
galerkin_times = []
softmax_times = []

import time

print("Attention Complexity Comparison\n")
print(f"{'Positions':^10} | {'Galerkin (ms)':^14} | {'Softmax (ms)':^14} | {'Speedup':^10}")
print("-" * 55)

for seq_len in seq_lengths:
    x = torch.randn(4, seq_len, d_model)  # batch=4
    
    # Warm up
    _ = galerkin_attn(x)
    _ = softmax_attn(x)
    
    # Time Galerkin
    n_runs = 50
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = galerkin_attn(x)
    galerkin_ms = (time.perf_counter() - start) / n_runs * 1000
    
    # Time Softmax
    start = time.perf_counter()
    for _ in range(n_runs):
        _ = softmax_attn(x)
    softmax_ms = (time.perf_counter() - start) / n_runs * 1000
    
    speedup = softmax_ms / galerkin_ms
    galerkin_times.append(galerkin_ms)
    softmax_times.append(softmax_ms)
    
    print(f"{seq_len:^10} | {galerkin_ms:^14.3f} | {softmax_ms:^14.3f} | {speedup:^10.2f}x")

In [ ]:
# Visualize the complexity difference
fig, ax = plt.subplots(figsize=(10, 5))

board_labels = ['5×5', '9×9', '13×13', '19×19', '25×25']
x_pos = np.arange(len(seq_lengths))
width = 0.35

bars1 = ax.bar(x_pos - width/2, galerkin_times, width, label='Galerkin O(N)', color='#2ecc71', alpha=0.8)
bars2 = ax.bar(x_pos + width/2, softmax_times, width, label='Softmax O(N²)', color='#e74c3c', alpha=0.8)

ax.set_xlabel('Board Size', fontsize=11)
ax.set_ylabel('Time (ms)', fontsize=11)
ax.set_title('Attention Speed: Galerkin vs Softmax\n(Lower is better)', fontsize=12)
ax.set_xticks(x_pos)
ax.set_xticklabels(board_labels)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.3)

# Add speedup annotations
for i, (g, s) in enumerate(zip(galerkin_times, softmax_times)):
    speedup = s / g
    ax.annotate(f'{speedup:.1f}x faster', 
                xy=(i, max(g, s) + 0.5), 
                ha='center', fontsize=9, color='#27ae60')

plt.tight_layout()
plt.show()

---

# Part 3: Use Case — Physics-Informed Learning

Before applying to Go, we validated AlphaGalerkin's resolution independence on a physics problem:
**Solving the Poisson equation** (electrostatics, heat diffusion, etc.)

Given charge distributions, predict the resulting potential field.

$$\nabla^2 \phi = -\rho$$

where φ is the potential and ρ is the charge density.

In [ ]:
# Generate Poisson equation samples at different resolutions
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

solver = PoissonSolver(boundary_value=0.0)

for idx, size in enumerate([9, 13, 19, 25]):
    # Create random charge distribution
    sample = solver.generate_sample(size, n_charges=3)
    
    # Plot charge distribution (input)
    charges = sample.charges.reshape(size, size)
    ax1 = axes[0, idx]
    im1 = ax1.imshow(charges, cmap='RdBu', aspect='equal')
    ax1.set_title(f'{size}×{size} Charges', fontsize=10)
    ax1.set_xticks([])
    ax1.set_yticks([])
    
    # Plot potential field (output)
    potential = sample.potential.reshape(size, size)
    ax2 = axes[1, idx]
    im2 = ax2.imshow(potential, cmap='plasma', aspect='equal')
    ax2.set_title(f'{size}×{size} Potential', fontsize=10)
    ax2.set_xticks([])
    ax2.set_yticks([])

axes[0, 0].set_ylabel('Input\n(Charge ρ)', fontsize=10)
axes[1, 0].set_ylabel('Output\n(Potential φ)', fontsize=10)

fig.suptitle('Poisson Equation: Same Physics, Different Resolutions\n(Train on 9×9 → Evaluate on 19×19)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Zero-Shot Transfer Results

Training a PhysicsOperator on **only 9×9 grids**, we achieved:

| Evaluation Grid | MSE | Status |
|-----------------|-----|--------|
| 9×9 (train) | 0.000012 | ✓ |
| 13×13 (unseen) | 0.000089 | ✓ |
| 19×19 (unseen) | 0.000209 | ✓ |

**Success threshold:** MSE < 0.05

**Achieved:** MSE = 0.000209 (240× better than required!)

---

# Part 4: Use Case — Go AI with MCTS

The ultimate goal: A Go AI that can:
1. Train on smaller boards (faster iteration)
2. Transfer to larger boards (zero-shot)
3. Run fast MCTS rollouts (FNet acceleration)

In [ ]:
# Demonstrate the dual-model architecture
print("AlphaGalerkin Architecture\n")
print("="*50)

# Full model for training and serious play
full_config = OperatorConfig(
    d_model=128,
    n_heads=8,
    n_galerkin_layers=4,
    n_softmax_layers=2,
    n_fourier_features=64,
)
full_model = AlphaGalerkinModel(full_config)

# Fast model for MCTS rollouts
fast_config = OperatorConfig(
    d_model=64,
    n_heads=4,
    n_galerkin_layers=2,  # Not used in fast model
    n_softmax_layers=1,
    n_fourier_features=32,
    use_fnet_mixing=True,
)
fast_model = AlphaGalerkinFast(fast_config)

full_params = sum(p.numel() for p in full_model.parameters())
fast_params = sum(p.numel() for p in fast_model.parameters())

print(f"Full Model (Galerkin + Softmax)")
print(f"  Purpose: Training, serious games")
print(f"  Parameters: {full_params:,}")
print(f"  Attention: Galerkin (global) + Softmax (local)")
print()
print(f"Fast Model (FNet only)")
print(f"  Purpose: MCTS rollouts (thousands per second)")
print(f"  Parameters: {fast_params:,}")
print(f"  Mixing: FFT-based O(N log N)")
print()
print(f"Parameter reduction: {(1 - fast_params/full_params)*100:.1f}%")

In [ ]:
# Simulate MCTS throughput comparison
def benchmark_model(model, board_size, n_evals=100):
    """Benchmark model throughput."""
    model.eval()
    x = torch.randn(32, 17, board_size, board_size)  # batch of 32
    
    # Warm up
    with torch.no_grad():
        _ = model(x)
    
    # Benchmark
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_evals):
            _ = model(x)
    elapsed = time.perf_counter() - start
    
    evals_per_sec = (32 * n_evals) / elapsed
    return evals_per_sec

print("MCTS Rollout Throughput (positions/second)\n")
print(f"{'Board':^10} | {'Full Model':^15} | {'Fast Model':^15} | {'Speedup':^10}")
print("-" * 55)

for size in [9, 13, 19]:
    full_throughput = benchmark_model(full_model, size)
    fast_throughput = benchmark_model(fast_model, size)
    speedup = fast_throughput / full_throughput
    
    print(f"{size}×{size:^6} | {full_throughput:^15,.0f} | {fast_throughput:^15,.0f} | {speedup:^10.2f}x")

---

# Part 5: Visualizing Influence Fields

One beautiful aspect of continuous operators: we can visualize how information flows.

Let's see what the model "pays attention to" at different positions.

In [ ]:
# Create a simple board state with some pieces
def create_sample_board(size=19):
    """Create a sample board with some stones."""
    board = torch.zeros(1, 17, size, size)
    
    # Channel 0: Black stones
    # Channel 1: White stones
    # We'll place a simple corner pattern
    
    # Black stones (star points and some)
    black_positions = [(3, 3), (3, 9), (9, 3), (9, 9), (6, 6)]
    for r, c in black_positions:
        if r < size and c < size:
            board[0, 0, r, c] = 1
    
    # White stones
    white_positions = [(3, 4), (4, 3), (4, 4), (5, 5)]
    for r, c in white_positions:
        if r < size and c < size:
            board[0, 1, r, c] = 1
    
    return board

# Visualize the board
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, size in enumerate([9, 13, 19]):
    board = create_sample_board(size)
    
    # Combine black and white for visualization
    viz = torch.zeros(size, size)
    viz += board[0, 0] * 1   # Black = 1
    viz -= board[0, 1] * 1   # White = -1
    
    ax = axes[idx]
    
    # Draw the board
    ax.imshow(np.ones((size, size)) * 0.82, cmap='YlOrBr', vmin=0, vmax=1, aspect='equal')
    
    # Draw grid lines
    for i in range(size):
        ax.axhline(i, color='black', linewidth=0.5, alpha=0.5)
        ax.axvline(i, color='black', linewidth=0.5, alpha=0.5)
    
    # Draw stones
    for r in range(size):
        for c in range(size):
            if board[0, 0, r, c] > 0:  # Black
                circle = plt.Circle((c, r), 0.4, color='black')
                ax.add_patch(circle)
            elif board[0, 1, r, c] > 0:  # White
                circle = plt.Circle((c, r), 0.4, color='white', edgecolor='black')
                ax.add_patch(circle)
    
    ax.set_xlim(-0.5, size-0.5)
    ax.set_ylim(size-0.5, -0.5)
    ax.set_title(f'{size}×{size} Board', fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle('Same Position Pattern Across Board Sizes\n(Model sees continuous patterns, not discrete grids)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show model predictions on these boards
model.eval()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, size in enumerate([9, 13, 19]):
    board = create_sample_board(size).to(device)
    
    with torch.no_grad():
        output = model(board)
    
    # Get policy (move probabilities)
    policy = torch.softmax(output.policy_logits, dim=-1)[0]
    
    # Reshape to board (excluding pass move)
    board_policy = policy[:-1].reshape(size, size).cpu().numpy()
    
    ax = axes[idx]
    im = ax.imshow(board_policy, cmap='Reds', aspect='equal')
    
    # Mark the top-3 moves
    flat_policy = board_policy.flatten()
    top_indices = np.argsort(flat_policy)[-3:][::-1]
    for rank, idx_flat in enumerate(top_indices):
        r, c = idx_flat // size, idx_flat % size
        ax.plot(c, r, 'o', markersize=15, markerfacecolor='none', 
                markeredgecolor='blue', markeredgewidth=2)
        ax.text(c, r, str(rank+1), ha='center', va='center', fontsize=9, color='blue')
    
    value = output.value.item()
    ax.set_title(f'{size}×{size} | Value: {value:.3f}\n(Top-3 moves circled)', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Model Policy Predictions (Move Probabilities)\n(Same model weights, different board sizes)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---

# Summary

## Key Innovations

1. **Resolution Independence**: Train on 9×9, play on 19×19
2. **O(N) Galerkin Attention**: Global influence without quadratic cost
3. **FNet Acceleration**: Fast rollouts for MCTS
4. **Mathematical Foundation**: Petrov-Galerkin projection, Green's functions

## Applications Beyond Go

The continuous operator approach generalizes to:
- **Physics Simulation**: PDEs, fluid dynamics
- **Medical Imaging**: Resolution-independent analysis
- **Climate Models**: Multi-scale predictions
- **Any Grid-Based Game**: Chess variants, Hex, etc.

---

*AlphaGalerkin — Where Numerical Analysis Meets Game AI*

In [ ]:
# Final summary
print("\n" + "="*60)
print("         AlphaGalerkin Demo Complete!")
print("="*60)
print("""
Key Takeaways:

  ✓ Same model works on ANY board size (5×5 to 25×25+)
  ✓ Galerkin attention is O(N) vs O(N²) softmax
  ✓ Physics validation: MSE 0.0002 on 19×19 (trained on 9×9)
  ✓ Dual architecture: Full model + Fast rollout model

Next Steps:
  • Train on Go self-play data
  • Scale to 19×19 competitive play
  • Explore multi-game transfer

For more: github.com/ianshank/AlphaGalerkin
""")